# **EKSTRAK LONGITUDE DAN LATITUDE MENGGUNAKAN HERE API**

In [19]:
import requests
import pandas as pd
import time

In [20]:
API_KEY = "ZlxPC6jHOSFfkREkwdKY8S1NLV61Su8niqKB0Dp259w"
BASE_URL = "https://geocode.search.hereapi.com/v1/geocode"

In [21]:
INPUT_FILE = "df_not_indonesia_raw.csv"
OUTPUT_FILE = "output_geocode_not_indonesian-v2.csv"

In [22]:
df = pd.read_csv(INPUT_FILE)

In [23]:
def geocode_address(address: str):
    if not isinstance(address, str) or not address.strip():
        return None, None

    params = {
        "q": address,
        "apiKey": API_KEY,
        "in": "countryCode:IDN"
    }

    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        items = data.get("items", [])
        if not items:
            return None, None

        pos = items[0].get("position", {})
        lat = pos.get("lat")
        lng = pos.get("lng")

        # tambahan verifikasi wilayah Indonesia
        if lat is None or lng is None:
            return None, None
        if not (-11 < lat < 7 and 95 < lng < 141):
            return None, None

        return lat, lng

    except Exception as e:
        print(f"Error untuk alamat '{address}': {e}")
        return None, None

In [24]:
latitudes = []
longitudes = []

for idx, row in df.iterrows():
    alamat = row.get("Alamat_lengkap", "")
    print(f"[{idx}] Geocode: {alamat}")

    lat, lng = geocode_address(alamat)
    latitudes.append(lat)
    longitudes.append(lng)

    time.sleep(0.2)

[0] Geocode: Kerinci Gunung Raya Lempur Tengah Jl. Raya Lempur Tengah Kecamatan Gunung Raya, Lempur Tengah, Gunung Raya, Kab. Kerinci, Jambi JAMBI
[1] Geocode: Kerinci Kayu Aro Mekar Jaya Jl. Lintas Sungai Tanduk, Mekar RT 06 JAMBI
[2] Geocode: Merangin Margo Tabir Suko Rejo Jl. Poros Margoyoso Hitam Ulu Suko Rejo Rt. 04 Rw. 01 Desa Suko Rejo, Kecamatan Margo Tabir, Kabupaten Merangin Provinsi Jambi JAMBI
[3] Geocode: Muaro Jambi Sungai Bahar Mekar Sari Makmur Simpang Yanto, RT 009, Desa Suka Makmur, Kecamatan Sungai Bahar, Kabupaten Muaro Jambi, Provinsi Jambi JAMBI
[4] Geocode: Rokan Hilir Bangko Pusako Bangko Permata Jl. Lintas Sumatera Km 7 Bangko Permata, Kec Bangko Pusako Rokan Hilir, Bangko Permata, Bangko Pusako, Kab. Rokan Hilir, Riau RIAU
[5] Geocode: Rokan Hilir Tanah Putih Banjar XII 2 Jl. Lintas Riau Sumut Kel. Banjar Xii Km 167 Kecamatan Tanah Putih Kabupaten Rohil, Riau RIAU
[6] Geocode: Rokan Hulu Tambusai Sungai Kumango Jl. Lintas Provinsi Riau-Sumatera Utara, Sungai K

In [25]:
df["Latitude"] = latitudes
df["Longitude"] = longitudes

In [26]:
df.to_csv(OUTPUT_FILE, index=False)
print(f"Selesai! Hasil disimpan ke file: {OUTPUT_FILE}")

Selesai! Hasil disimpan ke file: output_geocode_not_indonesian-v2.csv


In [27]:
# import pandas as pd
# df = pd.read_csv('Daftar SPPG_LongLat.csv')
# df.head()

In [28]:
# df.to_csv('Daftar SPPG_LongLat(2).csv', sep=';', index=False)

In [ ]:
import pandas as pd
import requests
import time

API_KEY = "ZlxPC6jHOSFfkREkwdKY8S1NLV61Su8niqKB0Dp259w"  # ganti punyamu

def get_province(lat, lon):
    url = f"https://geocode.search.hereapi.com/v1/revgeocode?at={lat},{lon}&apikey={API_KEY}"
    r = requests.get(url).json()
    try:
        return r["items"][0]["address"].get("state", None)
    except Exception:
        return None

# 1. Baca CSV (pakai ; kalau dari Excel lokal)
df = pd.read_csv("Daftar SPPG V3.csv", sep=";", engine="python")

# Cek dulu kolom apa saja yang ada
print(df.columns)

# 2. Sesuaikan nama kolom latitude & longitude di sini
LAT_COL = "LAT"       # ganti sesuai nama kolom di CSV kamu
LON_COL = "LON"       # misal "LNG", "LONGITUDE", dsb

# 3. Tambah kolom province
provinces = []
for i, row in df.iterrows():
    lat = row[LAT_COL]
    lon = row[LON_COL]
    province = get_province(lat, lon)
    provinces.append(province)
    time.sleep(0.2)  # biar nggak kena rate limit

df["province"] = provinces

# 4. Simpan ke file baru
df.to_csv("Daftar_SPPG_dengan_provinsi.csv", index=False)
print("Selesai! File tersimpan sebagai: Daftar_SPPG_dengan_provinsi.csv")